# SynBioCrow 2.2.0 RC2 — published-artifact acceptance
Validates the **published GitHub RC2 assets** independently of the development checkout. A green run is the promotion gate for final `v2.2.0`.


In [ ]:
import os,sys,json,hashlib,subprocess,shutil,zipfile,urllib.request
from pathlib import Path
from google.colab import drive
OWNER="theiman112860"; REPO="SynBioCrow"; TAG="v2.2.0-rc2"
EXPECTED_COMMIT="631bac03b9a930a29721b955d1a06515942df23c"
ASSETS={
"SynBioCrow_2.2.0-rc2_PACKAGE_SUMMARY.json":"b06b154a5199418b2d5c34fa251aa8c8191b00a95e05753f771a08d796db819d",
"SynBioCrow_2.2.0-rc2_release_manifest.json":"d497fd5f432a0c1c85983aa7f2ea657606e700f377afe72ffac6af2fd8d4127e",
"SynBioCrow_2.2.0-rc2_SOURCE_REPRODUCIBILITY.zip":"cacec275839f4fccb36f5c6732ab99d4c132c92c09e89c6ce3df86c86687dca5"}
ROOT=Path("/content/synbiocrow_rc2_acceptance")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir()
def sha256(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
 return h.hexdigest()
def run(cmd,cwd=None,timeout=1800,env=None):
 print("$"," ".join(map(str,cmd)),flush=True)
 p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout,env=env)
 print(p.stdout,flush=True)
 if p.returncode: raise RuntimeError(f"command failed rc={p.returncode}: {cmd}")
 return p.stdout
print("Acceptance target",TAG,EXPECTED_COMMIT)


In [ ]:
# Download the actual published release assets and verify their GitHub-recorded digests.
for name,expected in ASSETS.items():
 url=f"https://github.com/{OWNER}/{REPO}/releases/download/{TAG}/{name}"
 dst=ROOT/name
 print("download",url)
 urllib.request.urlretrieve(url,dst)
 got=sha256(dst)
 print(name,dst.stat().st_size,got)
 assert got==expected,(name,got,expected)
print("PUBLISHED RC2 ASSET DIGESTS: PASS")


In [ ]:
# Verify package/manifest commit equality and every packaged file hash.
summary=json.loads((ROOT/"SynBioCrow_2.2.0-rc2_PACKAGE_SUMMARY.json").read_text())
manifest=json.loads((ROOT/"SynBioCrow_2.2.0-rc2_release_manifest.json").read_text())
assert summary["git_commit"]==EXPECTED_COMMIT,summary["git_commit"]
assert manifest["git_commit"]==EXPECTED_COMMIT,manifest["git_commit"]
zip_path=ROOT/summary["source_zip"]
assert sha256(zip_path)==summary["source_zip_sha256"]
src=ROOT/"source"; src.mkdir()
with zipfile.ZipFile(zip_path) as z:z.extractall(src)
bad=[]
for item in manifest["files"]:
 p=src/item["path"]
 if not p.is_file() or p.stat().st_size!=item["bytes"] or sha256(p)!=item["sha256"]:bad.append(item["path"])
assert not bad,bad[:20]
print("PACKAGE/TAG COMMIT MATCH: PASS")
print("M9 MANIFEST INVENTORY: PASS",len(manifest["files"]),"files")


In [ ]:
# Clean installation into a dedicated target directory; no Colab venv dependency.
site=ROOT/"isolated_site"; site.mkdir()
run([sys.executable,"-m","pip","install","--upgrade","--target",site,str(src)])
def iso(code,cwd=None,timeout=1800):
 return run([sys.executable,"-I","-c",f"import sys; sys.path.insert(0,{str(site)!r});\n"+code],cwd=cwd,timeout=timeout)
iso("import synbiocrow; print('isolated import PASS',synbiocrow.__file__)")
iso("from importlib.metadata import version; print('installed version',version('synbiocrow'))")
print("PUBLISHED RC2 CLEAN INSTALL: PASS")


In [ ]:
# Run all packaged tests against the isolated published artifact.
run([sys.executable,"-m","pip","install","--upgrade","--target",site,"pytest"])
tests=src/"tests"
assert tests.is_dir(),"published RC2 must contain tests"
iso(f"import pytest; raise SystemExit(pytest.main(['-q',{str(tests)!r}]))",cwd=src)
print("PUBLISHED RC2 TEST SUITE: PASS")


In [ ]:
# Run M7 and M9 check from the published artifact, importing the isolated install first.
iso(f"import runpy,sys; sys.argv=['m7_release_readiness.py']; runpy.run_path({str(src/'scripts/m7_release_readiness.py')!r},run_name='__main__')",cwd=src)
iso(f"import runpy,sys; sys.argv=['build_m9_release.py','--check']; runpy.run_path({str(src/'scripts/build_m9_release.py')!r},run_name='__main__')",cwd=src)
print("PUBLISHED RC2 M7 + M9: PASS")


In [ ]:
# Audit the public release contract and certified KNIME-free RetroPath evidence.
readme=(src/"README.md").read_text()
assert "2.2.0-rc2" in readme
assert "retropath standalone" in readme.lower()
assert ("not required" in readme.lower()) or ("optional legacy" in readme.lower())
assert "biopks" in readme.lower()
assert all(x.lower() in readme.lower() for x in ["Candidate","Mature","Certified"])
cert=(src/"docs/RETROPATH_STANDALONE_EQUIVALENCE.md").read_text()
for x in ["EXACT_COMPOUND_TRANSITION_MULTISET","historical rows: 34","standalone rows: 34","precision: 1.0","recall: 1.0","Jaccard: 1.0","KNIME is not required"]:
 assert x.lower() in cert.lower(),x
print("PUBLIC CONTRACT + RETROPATH CERTIFICATE: PASS")


In [ ]:
# Persist final acceptance evidence.
report={"schema":"synbiocrow.rc_acceptance.v1","release":"2.2.0-rc2","tag":TAG,"commit":EXPECTED_COMMIT,
"published_asset_sha256":ASSETS,"source_zip_sha256":sha256(zip_path),"source_zip_bytes":zip_path.stat().st_size,
"manifest_file_count":manifest["file_count"],"m7_reproducibility_digest":manifest["m7_reproducibility_digest"],
"retropath_primary_runtime":manifest["retropath_primary_runtime"],"retropath_knime_required":manifest["retropath_knime_required"],
"retropath_equivalence_policy":manifest["retropath_equivalence_policy"],
"checks":{"asset_digests":"PASS","package_tag_commit_match":"PASS","manifest_inventory":"PASS","clean_install":"PASS","tests":"PASS","m7":"PASS","m9":"PASS","public_contract":"PASS","retropath_certificate":"PASS"},
"acceptance":"PASS","promotion_candidate":"v2.2.0"}
rp=ROOT/"SynBioCrow_2.2.0-rc2_ACCEPTANCE_REPORT.json"; rp.write_text(json.dumps(report,indent=2,sort_keys=True)+"\n")
bundle=ROOT/"SynBioCrow_2.2.0-rc2_ACCEPTANCE_BUNDLE.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
 for p in [rp,ROOT/"SynBioCrow_2.2.0-rc2_PACKAGE_SUMMARY.json",ROOT/"SynBioCrow_2.2.0-rc2_release_manifest.json"]:z.write(p,p.name)
print(json.dumps(report,indent=2))
print("report sha256",sha256(rp));print("bundle sha256",sha256(bundle))
drive.mount("/content/drive",force_remount=False)
out=Path("/content/drive/MyDrive/SynBioCrow/release/2.2.0-rc2/acceptance");out.mkdir(parents=True,exist_ok=True)
for p in [rp,bundle]:shutil.copy2(p,out/p.name);print("saved",out/p.name)
print("RC2 PUBLISHED-ARTIFACT ACCEPTANCE: PASS")
print("PROMOTION GATE: READY FOR FINAL v2.2.0")
